# RealMLP (reused) + external SDSS17 — OOF generator

Reuses the proven **RealMLP-TD** architecture from the public `nb02` solution verbatim, but:
1. trains on the competition data **plus the external SDSS17 dataset** (cleaned of `-9999`
   sentinels), which the public solutions do not use;
2. uses the **same 5-fold split** as `modeling_note.ipynb`
   (`StratifiedKFold(5, shuffle=True, random_state=42)` over the real rows, external rows
   appended to training only and excluded from OOF) so the saved OOF aligns row-by-row for
   downstream stacking;
3. saves `oof_mlp.npy` / `test_mlp.npy` / `oof_y.npy` for `ensemble_note.ipynb`.

Heavy: ~3 h on a Kaggle GPU. Run once, reuse the saved arrays.

## GPU bootstrap

In [ ]:
import subprocess, sys
def _gpu_name():
    try:
        return subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True).strip()
    except Exception:
        return ''
_NAME = _gpu_name()
print('Detected GPU:', _NAME or '(none / CPU)')
# Pascal P100 = compute capability 6.0, unsupported by the cu128 wheel.
if 'P100' in _NAME:
    print('P100 detected -> installing Pascal-compatible torch (cu121) ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch==2.5.1', '--index-url',
                    'https://download.pytorch.org/whl/cu121'], check=False)
import torch
# verify the GPU can actually launch a kernel; otherwise fall back to CPU
CUDA_OK = False
if torch.cuda.is_available():
    try:
        (torch.zeros(8, device='cuda') + 1).sum().item()
        CUDA_OK = True
        print('CUDA kernel test OK on', torch.cuda.get_device_name(0))
    except Exception as e:
        print('WARNING: CUDA kernel test failed, using CPU:', e)
print('torch', torch.__version__, '| CUDA_OK', CUDA_OK)

## Imports

In [ ]:
import math, random, warnings, os, glob, gc
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings('ignore')

def seed_everything(s):
    np.random.seed(s); random.seed(s); torch.manual_seed(s)
SEED = 42; FOLDS = 5; n_classes = 3
seed_everything(SEED)
ID, TARGET = 'id', 'class'
CLASSES = ['GALAXY', 'QSO', 'STAR']
USE_EXTERNAL = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device', device)

## Load data + external SDSS17

Real competition rows come first; external rows (cleaned) are appended after, so a
`StratifiedKFold` over the first `n_real` rows reproduces the split used by the booster
notebook exactly.

In [ ]:
def find_path():
    cands = (glob.glob('/kaggle/input/*/train.csv')
             + glob.glob('/kaggle/input/competitions/*/train.csv')
             + ['../docs/dataset/train.csv'])
    return next(os.path.dirname(c) + '/' for c in cands if os.path.exists(c))

PATH = find_path()
train = pd.read_csv(PATH + 'train.csv')
test  = pd.read_csv(PATH + 'test.csv')
train[TARGET] = train[TARGET].map({c: i for i, c in enumerate(CLASSES)})
n_real = len(train)
print('real train', train.shape, '| test', test.shape)

# --- external SDSS17 (cleaned), appended after the real rows ---
n_ext = 0
if USE_EXTERNAL:
    ext_paths = sorted(Path('/kaggle/input').rglob('star_classification.csv'))
    if not ext_paths:
        local = Path(PATH).parent.parent / 'external' / 'star_classification.csv'
        if local.exists(): ext_paths = [local]
    if ext_paths:
        ext = pd.read_csv(ext_paths[0])
        ext[TARGET] = ext['class'].astype(str).str.upper().map({c: i for i, c in enumerate(CLASSES)})
        ext = ext[~(ext[['u','g','r','i','z']] < -50).any(axis=1)].copy()   # drop -9999 sentinels
        ext = ext.dropna(subset=[TARGET])
        for c in ['spectral_type', 'galaxy_population']:
            if c not in ext.columns: ext[c] = np.nan
        common = [c for c in train.columns if c in ext.columns]
        ext = ext[common].copy()
        ext[ID] = -1
        n_ext = len(ext)
        train = pd.concat([train, ext], ignore_index=True)
        print('appended external rows', n_ext, '| combined train', train.shape)
    else:
        print('external not found — skipping')

y_all = train[TARGET].values.astype(int)
test_id = test[ID].values
real_idx = np.arange(n_real)              # real rows are the first n_real
ext_idx  = np.arange(n_real, len(train))  # external rows
y_real = y_all[real_idx]

## Astronomy feature engineering (from nb02)

In [ ]:
BANDS = ['u', 'g', 'r', 'i', 'z']
def add_features(df):
    for a, b in [('u','g'),('g','r'),('r','i'),('i','z'),
                 ('u','r'),('u','z'),('g','i'),('g','z'),('r','z'),('u','i')]:
        df[f'{a}_{b}'] = df[a] - df[b]
    M = df[BANDS].values
    df['mag_mean'] = M.mean(1); df['mag_std'] = M.std(1)
    df['mag_min'] = M.min(1); df['mag_max'] = M.max(1)
    z = df['redshift'].values
    df['z_log1p'] = np.log1p(np.clip(z, 0, None))
    df['z_clip']  = np.clip(z, -0.01, 7.0)
    return df

train = add_features(train)
test  = add_features(test)
X = train.drop([ID, TARGET], axis=1)
X_test = test.drop([ID], axis=1)
del train, test; gc.collect()

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print('init cats', len(cat_cols), 'nums', len(num_cols))

## Preprocess: floor-categorize + interaction combos (from nb02)

In [ ]:
X = train.drop([ID, TARGET], axis=1)
X_test = test.drop([ID], axis=1)
del train, test; gc.collect()

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print('init cats', len(cat_cols), 'nums', len(num_cols))

category_map = {}
important_combos = sorted([('alpha_cat_', 'delta_cat_'), ('u_cat_', 'z_cat_')])

def feature_engineering(df, fit=False):
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize(); category_map[col] = uniques
        else:
            code_map = {c: i for i, c in enumerate(category_map[col])}
            codes = df[col].map(code_map).fillna(-1).astype('int32')
        df[col] = codes; df[col] = df[col].astype('category')
    for col in num_cols:
        cat_name = f'{col}_cat_'
        if fit:
            codes, uniques = np.floor(df[col]).factorize(); category_map[col] = uniques
        else:
            code_map = {c: i for i, c in enumerate(category_map[col])}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes; df[cat_name] = df[cat_name].astype('category')
    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'; combo_names.append(combo_name)
        s = df[cols[0]].astype(str)
        for col in cols[1:]:
            s = s + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(s, sort=False); category_map[combo_name] = uniques
        else:
            code_map = {c: i for i, c in enumerate(category_map[combo_name])}
            codes = s.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes; df[combo_name] = df[combo_name].astype('category')
    new_cat = [c for c in df.columns if c.endswith('_')]
    return df, new_cat, combo_names

X, new_cat, combo_names = feature_engineering(X, fit=True)
X_test, _, _ = feature_engineering(X_test, fit=False)
cat_cols = sorted(cat_cols + new_cat)
X = X.reindex(sorted(X.columns), axis=1)
X_test = X_test.reindex(sorted(X_test.columns), axis=1)
print('final cats', len(cat_cols), 'total cols', X.shape[1])

## RealMLP model components (verbatim from nb02)

In [ ]:
class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, tfms):
        self._tfms = [t for t in tfms if t in ("median_center","robust_scale","smooth_clip","l2_normalize")]
    def fit(self, X, y=None):
        if "median_center" in self._tfms or "robust_scale" in self._tfms:
            self._median = np.median(X, axis=0)
            q = np.quantile(X,0.75,axis=0) - np.quantile(X,0.25,axis=0)
            zi = q == 0.0
            q[zi] = 0.5*(X.max(0)[zi]-X.min(0)[zi])
            self._iqr = 1.0/(q+1e-30); self._iqr[q==0.0] = 0.0
        return self
    def transform(self, X, y=None):
        X = X.copy().astype(np.float32)
        for t in self._tfms:
            if t=="median_center": X -= self._median[None,:]
            elif t=="robust_scale": X *= self._iqr[None,:]
            elif t=="smooth_clip": X = X/np.sqrt(1+(X/3)**2)
            elif t=="l2_normalize":
                n = np.linalg.norm(X,axis=1,keepdims=True); X /= np.where(n==0,1.0,n)
        return X

class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens, cat_dims, embed_dim=8, onehot_thresh=8, device=None):
        super().__init__()
        self.n_ens=n_ens; self.cat_dims=cat_dims; self.onehot_features=[]
        self.embed_layers=nn.ModuleList(); self._embed_feature_indices=[]
        for i,dim in enumerate(cat_dims):
            if dim<=onehot_thresh: self.onehot_features.append(i)
            else:
                self.embed_layers.append(nn.ModuleList([nn.Embedding(dim,embed_dim) for _ in range(n_ens)]))
                self._embed_feature_indices.append(i)
    def forward(self,x):
        b,n_ens,_=x.shape; feats=[]
        if self.onehot_features:
            ox=x[:,:,self.onehot_features]; od=[self.cat_dims[i] for i in self.onehot_features]
            enc=torch.zeros(b,n_ens,sum(od),device=x.device); st=0
            for idx,dim in enumerate(od):
                enc.scatter_(2, ox[:,:,idx:idx+1].long()+st, 1.0); st+=dim
            feats.append(enc)
        for emb_list,fi in zip(self.embed_layers,self._embed_feature_indices):
            fe=[emb_list[mi](x[:,mi,fi:fi+1].long()) for mi in range(self.n_ens)]
            feats.append(torch.cat(fe,dim=1))
        return torch.cat(feats,dim=2)

class ScalingLayer(nn.Module):
    def __init__(self,n_ens,n_features):
        super().__init__(); self.scale=nn.Parameter(torch.ones(n_ens,n_features))
    def forward(self,x): return x*self.scale[None,:,:]

class NTPLinear(nn.Module):
    def __init__(self,n_ens,in_f,out_f,bias=True):
        super().__init__(); self.in_features=in_f
        self.weight=nn.Parameter(torch.randn(n_ens,in_f,out_f))
        self.bias=nn.Parameter(torch.randn(n_ens,out_f)) if bias else None
    def forward(self,x):
        x=torch.einsum("bki,kio->bko",x,self.weight)/math.sqrt(self.in_features)
        if self.bias is not None: x=x+self.bias
        return x

class PBLDEmbedding(nn.Module):
    def __init__(self,n_ens,n_features,hidden_dim=16,out_dim=4,freq_scale=0.1,activation=nn.GELU):
        super().__init__(); self.out_dim=out_dim
        self.w1=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim)*freq_scale)
        self.b1=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim))
        self.w2=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim,out_dim-1)/math.sqrt(hidden_dim))
        self.b2=nn.Parameter(torch.zeros(n_ens,n_features,out_dim-1))
        self.act=activation(); nn.init.uniform_(self.b1,-math.pi,math.pi)
    def forward(self,x):
        periodic=torch.cos(2*math.pi*(x.unsqueeze(-1)*self.w1.unsqueeze(0)+self.b1.unsqueeze(0)))
        transformed=self.act(torch.einsum("bkfh,kfhd->bkfd",periodic,self.w2)+self.b2.unsqueeze(0))
        feat=torch.cat([x.unsqueeze(-1),transformed],dim=-1)
        return feat.flatten(start_dim=2)

class RealMLP(nn.Module):
    def __init__(self,output_dim,cat_dims,n_numerical,cfg):
        super().__init__(); n_ens=cfg["n_ens"]; embed_dim=cfg["embed_dim"]; self.n_ens=n_ens
        self.cate=CategoricalFeatureLayer(n_ens,cat_dims,embed_dim,cfg["onehot_thresh"])
        self.num_embed=PBLDEmbedding(n_ens,n_numerical,cfg["pbld_hidden_dim"],cfg["pbld_out_dim"],cfg["pbld_freq_scale"],cfg["pbld_activation"])
        num_emb=n_numerical*cfg["pbld_out_dim"]
        cat_emb=sum(c if c<=cfg["onehot_thresh"] else embed_dim for c in cat_dims)
        total=num_emb+cat_emb; act=cfg["activation"]; layers=[]
        if cfg["add_front_scale"]: layers.append(ScalingLayer(n_ens,total))
        self._dropout_modules=[]; in_dim=total
        for i,h in enumerate(cfg["hidden_dims"]):
            lin=NTPLinear(n_ens,in_dim,h)
            if i==0: self.first_linear=lin
            d=nn.Dropout(cfg["dropout"]); self._dropout_modules.append(d)
            layers+=[lin,act(),d]; in_dim=h
        self.hidden=nn.Sequential(*layers)
        self.output_layer=NTPLinear(n_ens,in_dim,output_dim)
    def forward(self,x_num,x_cat):
        x_num=x_num.unsqueeze(1).expand(-1,self.n_ens,-1)
        x_cat=x_cat.unsqueeze(1).expand(-1,self.n_ens,-1)
        x=torch.cat([self.num_embed(x_num),self.cate(x_cat)],dim=2)
        return F.softmax(self.output_layer(self.hidden(x)),dim=2)

def apply_schedule(v,p,sched,flat=0.3):
    if sched=="constant": return v
    if sched=="cos": return v*(math.cos(math.pi*p)+1)/2
    if sched=="flat_cos":
        if p<flat: return v
        t=(p-flat)/(1-flat); return v*(math.cos(math.pi*t)+1)/2
    if sched=="flat_anneal":
        if p<flat: return v
        t=(p-flat)/(1-flat); return v*(1-t)
    if sched=="sqrt_cos": return v*math.sqrt((math.cos(math.pi*p)+1)/2)
    if sched=="expm4t": return v*math.exp(-4*p)
    raise ValueError(sched)

def get_parameter_groups(model,p):
    fid=id(model.first_linear.weight); sc,pb,fw,ow,bi=[],[],[],[],[]
    for n,pa in model.named_parameters():
        if "num_embed" in n: pb.append(pa)
        elif "scale" in n: sc.append(pa)
        elif id(pa)==fid: fw.append(pa)
        elif "bias" in n: bi.append(pa)
        else: ow.append(pa)
    LR,WD=p["lr"],p["weight_decay"]
    return [
        {"params":sc,"lr":LR*p["lr_scale_mult"],"weight_decay":WD*p["wd_scale_mult"]},
        {"params":pb,"lr":LR*p["pbld_lr_factor"],"weight_decay":WD},
        {"params":fw,"lr":LR*p["first_layer_lr_factor"],"weight_decay":WD*p["first_layer_wd_factor"]},
        {"params":ow,"lr":LR,"weight_decay":WD},
        {"params":bi,"lr":LR*p["lr_bias_mult"],"weight_decay":WD*p["wd_bias_mult"]},
    ]

def smooth_ce_loss(yt,yp,ls=0.0,cw=None):
    nc=yp.size(1); ys=torch.full_like(yp,ls/nc)
    ys.scatter_(1,yt.unsqueeze(1),1.0-ls+ls/nc)
    loss=-(ys*torch.log(yp.clamp(1e-15,1))).sum(1)
    if cw is not None:
        sw=cw[yt]; return (loss*sw).sum()/sw.sum()
    return loss.mean()

In [ ]:
class RealMLP_TD_Classifier(BaseEstimator):
    def __init__(self, **kw): self.params={**CONFIG, **kw}
    def fit(self, Xtr_df, ytr, Xva_df, yva, cat_col_names=None, ckpt_path="ck.pth", X_test=None):
        p=self.params; dev=torch.device(p["device"] if torch.cuda.is_available() else "cpu")
        cat_col_names=cat_col_names or []
        num_col_names=[c for c in Xtr_df.columns if c not in cat_col_names]
        Xtn=Xtr_df[num_col_names].values.astype(np.float32); Xvn=Xva_df[num_col_names].values.astype(np.float32)
        Xtc=Xtr_df[cat_col_names].values.astype(np.int64); Xvc=Xva_df[cat_col_names].values.astype(np.int64)
        y_tr=np.asarray(ytr); y_v=np.asarray(yva)
        self.preprocessor_=NumericalPreprocessor(p["tfms"]).fit(Xtn)
        Xtn=self.preprocessor_.transform(Xtn); Xvn=self.preprocessor_.transform(Xvn)
        self.cat_col_names_=cat_col_names; self.num_col_names_=num_col_names
        if cat_col_names:
            allc=[Xtc,Xvc]
            if X_test is not None: allc.append(X_test[cat_col_names].values.astype(np.int64))
            cat_dims=(np.concatenate(allc,0).max(0)+1).tolist()
        else: cat_dims=[]
        self.cat_dims_=cat_dims
        if cat_dims:
            cm=np.array(cat_dims)-1; Xtc=np.clip(Xtc,0,cm); Xvc=np.clip(Xvc,0,cm)
        classes=np.unique(y_tr); self.classes_=classes
        cw=torch.as_tensor(compute_class_weight("balanced",classes=classes,y=y_tr),dtype=torch.float32,device=dev)
        self.model_=RealMLP(len(classes),cat_dims,Xtn.shape[1],p).to(dev)
        groups=get_parameter_groups(self.model_,p)
        for g in groups: g["lr_base"]=g["lr"]
        opt=torch.optim.AdamW(groups,betas=(p["mom"],p["sq_mom"]))
        Xtn=torch.as_tensor(Xtn,dtype=torch.float32,device=dev); Xtc=torch.as_tensor(Xtc,dtype=torch.long,device=dev)
        ytt=torch.as_tensor(y_tr,dtype=torch.long,device=dev)
        Xvn=torch.as_tensor(Xvn,dtype=torch.float32,device=dev); Xvc=torch.as_tensor(Xvc,dtype=torch.long,device=dev)
        n_ens=p["n_ens"]; tb=p["train_bs"]; eb=p["eval_bs"]; ep=p["epochs"]
        total=ep*len(y_tr); order=np.arange(len(y_tr)); nc=len(classes)
        best=-np.inf; best_ep=0; self.best_val_probs_=None
        for epoch in range(ep):
            self.model_.train()
            for s in range(0,len(y_tr),tb):
                prog=(epoch*len(y_tr)+s)/total; idx=order[s:s+tb]
                for g in opt.param_groups: g["lr"]=apply_schedule(g["lr_base"],prog,p["lr_sched"],p["flat_ratio"])
                opt.zero_grad(); yp=self.model_(Xtn[idx],Xtc[idx])
                ls=apply_schedule(p["ls_eps"],prog,p["ls_eps_sched"],p["flat_ratio"])
                dr=apply_schedule(p["dropout"],prog,p["p_drop_sched"],p["flat_ratio"])
                for dm in self.model_._dropout_modules: dm.p=dr
                loss=smooth_ce_loss(ytt[idx].repeat_interleave(n_ens),yp.reshape(-1,nc),ls=ls,cw=cw)
                loss.backward(); torch.nn.utils.clip_grad_norm_(self.model_.parameters(),p["grad_clip"]); opt.step()
            np.random.shuffle(order)
            self.model_.eval()
            with torch.no_grad():
                vp=np.concatenate([self.model_(Xvn[s:s+eb],Xvc[s:s+eb]).mean(1).cpu().numpy() for s in range(0,len(y_v),eb)],0)
            sc=balanced_accuracy_score(y_v,vp.argmax(1))
            if sc>best:
                best=sc; best_ep=epoch+1; self.best_val_probs_=vp.copy(); torch.save(self.model_.state_dict(),ckpt_path)
            if p["verbosity"]>=2: print(f"   epoch {epoch+1}/{ep}  ba={sc:.5f}  best={best:.5f}")
        self.model_.load_state_dict(torch.load(ckpt_path)); self.best_score_=best; self._dev=dev
        print(f"   -> best ba {best:.5f} (epoch {best_ep})"); return self
    def predict_proba(self, X):
        eb=self.params["eval_bs"]
        Xn=self.preprocessor_.transform(X[self.num_col_names_].values.astype(np.float32))
        Xc=np.clip(X[self.cat_col_names_].values.astype(np.int64),0,np.array(self.cat_dims_)-1)
        Xn=torch.as_tensor(Xn,dtype=torch.float32,device=self._dev); Xc=torch.as_tensor(Xc,dtype=torch.long,device=self._dev)
        self.model_.eval()
        with torch.no_grad():
            return np.concatenate([self.model_(Xn[s:s+eb],Xc[s:s+eb]).mean(1).cpu().numpy() for s in range(0,len(Xn),eb)],0)

## Configuration (verbatim from nb02)

In [ ]:
CONFIG = {
    "n_ens": 8, "embed_dim": 7, "onehot_thresh": 10,
    "hidden_dims": [512, 512, 512], "dropout": 0.05, "p_drop_sched": "expm4t",
    "activation": nn.SiLU, "add_front_scale": True,
    "pbld_hidden_dim": 20, "pbld_out_dim": 5, "pbld_freq_scale": 5.0,
    "pbld_activation": nn.PReLU, "pbld_lr_factor": 0.093,
    "lr": 0.01, "mom": 0.9, "sq_mom": 0.98, "lr_sched": "flat_cos", "flat_ratio": 0.3,
    "first_layer_lr_factor": 1.0, "first_layer_wd_factor": 0.1,
    "lr_scale_mult": 10.0, "lr_bias_mult": 0.1, "weight_decay": 0.013,
    "wd_scale_mult": 0.1, "wd_bias_mult": 0.5, "grad_clip": 1.0,
    "ls_eps": 0.04, "ls_eps_sched": "cos",
    "tfms": ["median_center", "robust_scale"],
    "epochs": 8, "train_bs": 256, "eval_bs": 10240, "verbosity": 2,
    "device": "cuda" if CUDA_OK else "cpu", "random_state": 42,
}
FOLDS, SEED = 5, 42
n_classes = 3

## 5-fold RealMLP — external in training, OOF on real rows only

Identical folds to the booster notebook. External rows join every training fold but never
appear in validation, so `oof_mlp` is a clean OOF over the real competition rows.

In [ ]:
skf = StratifiedKFold(FOLDS, shuffle=True, random_state=SEED)
oof_mlp = np.zeros((n_real, n_classes))
tst_mlp = np.zeros((len(X_test), n_classes))

for fold, (tr_pos, va_pos) in enumerate(skf.split(real_idx, y_real), 1):
    print(f"\n##### fold {fold}/{FOLDS} #####")
    tr_rows = np.concatenate([real_idx[tr_pos], ext_idx])   # real-train + ALL external
    va_rows = real_idx[va_pos]                              # real validation only
    X_tr, X_val, X_tst = X.iloc[tr_rows].copy(), X.iloc[va_rows].copy(), X_test.copy()

    # OOF multiclass target encoding on the interaction combos
    enc = TargetEncoder(target_type='multiclass', cv=FOLDS, smooth='auto',
                        shuffle=True, random_state=SEED)
    tr_e = enc.fit_transform(X_tr[combo_names], y_all[tr_rows])
    va_e = enc.transform(X_val[combo_names]); te_e = enc.transform(X_tst[combo_names])
    te_names = [f"_te_{col}_{c}" for col in combo_names for c in range(n_classes)]
    X_tr[te_names] = tr_e; X_val[te_names] = va_e; X_tst[te_names] = te_e

    model = RealMLP_TD_Classifier(**CONFIG)
    model.fit(X_tr, y_all[tr_rows], X_val, y_all[va_rows],
              cat_col_names=cat_cols, ckpt_path=f"mlp_f{fold}.pth", X_test=X_tst)
    oof_mlp[va_pos] = model.best_val_probs_
    tst_mlp += model.predict_proba(X_tst) / FOLDS
    print(f"   fold mlp ba={balanced_accuracy_score(y_real[va_pos], oof_mlp[va_pos].argmax(1)):.5f}")
    torch.cuda.empty_cache(); gc.collect()

print('\nRealMLP OOF BA:', round(balanced_accuracy_score(y_real, oof_mlp.argmax(1)), 5))
print(confusion_matrix(y_real, oof_mlp.argmax(1)))

## Save OOF / test probabilities

In [ ]:
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('../data_processed')
OUT.mkdir(parents=True, exist_ok=True)
np.save(OUT / 'oof_mlp.npy', oof_mlp.astype('float32'))
np.save(OUT / 'test_mlp.npy', tst_mlp.astype('float32'))
np.save(OUT / 'oof_y.npy', y_real.astype('int8'))
np.save(OUT / 'test_id.npy', test_id)
print('saved oof_mlp', oof_mlp.shape, '| test_mlp', tst_mlp.shape, 'to', OUT.resolve())